# VEP Research - methods landscape and where we can go next
## Variant Effect Prediction for nicotinic acetylcholine receptors (nAChRs)

*Date: 2026-06-14*

**What this notebook answers** (your three questions):
1. **Is "VEP" just a name for "do some ML in the back", i.e. does it depend only on the algorithm - or is there more to it?**
2. **Can we try other / newer / better / more precise VEP methods** than the engineered-features + classifier recipe we inherited from the VEP-ENaC project?
3. **How should we handle the new papers you found** so I can read them and judge whether we can do something new?

**The headline answer up front.** VEP is a *task*, not a single algorithm, and the algorithm is usually the *least* important part. More importantly, our project is a slightly unusual VEP: we predict **gain- vs loss-of-function (GOF/LOF)** - the *direction* of effect - which most famous tools (AlphaMissense, ESM, PolyPhen) do **not** do. There is a small, recent, and very relevant family of methods that *do* exactly our task (funNCion, LoGoFunc, PreMode, and the Oct-2025 ion-channel paper **MissION**), and they point clearly at how to make ours better. Details below.

> Style note: like `week1.ipynb`, this is a written brief (no code to run). Plain-language explanations, with citations and links collected at the end.


## 1. Is VEP "just the ML algorithm"? No - here is what it actually is

**VEP = Variant Effect Prediction is a *task*, not a method.** The task is: given a genetic variant (here, an amino-acid change in an nAChR subunit), predict its effect. *How* you do that is wide open - and historically the biggest improvements have come from **almost everywhere except the choice of classifier.**

Any VEP system has **five parts**. The algorithm is only one of them, and rarely the decisive one:

```
 A variant  (e.g. CHRNA1 p.Glu204Lys)
        |
        v
 [ 1. TARGET ]        What are we predicting?
                      LOF vs GOF (us) | benign vs pathogenic | a continuous fitness score
        |
        v
 [ 2. DATA ]          Labelled variants
                      our 351 curated | ClinVar | deep mutational scanning (DMS)
        |
        v
 [ 3. REPRESENTATION ] Turn the variant into numbers  <-- where most of the "magic" lives
                      physicochemical | structural | evolutionary conservation | language-model embeddings
        |
        v
 [ 4. MODEL ]         The ML algorithm
                      logistic regression | XGBoost | neural net | a zero-shot language model
        |
        v
 [ 5. EVALUATION ]    Honest measurement
                      grouped cross-validation | AUC | calibration | sensible baselines
        |
        v
   A prediction  (and, ideally, how much to trust it)
```

**Why "it is mostly the algorithm" is a myth.** The last decade's jumps in VEP did **not** come from swapping one classifier for another. They came from **better representations and training paradigms**:
- **SIFT / PolyPhen-2** (2000s): conservation + a few features + a simple model.
- **EVE / DeepSequence** (2018-2021): learn a generative model of a protein family from evolution - no variant labels needed.
- **ESM-1v / ESM-2** (2021-2023): a protein *language model* reads millions of sequences and can score a mutation "zero-shot".
- **AlphaMissense** (2023): combines a structure model (AlphaFold) + a language model + weak labels.

In every case the **representation** changed, not the downstream maths. So when you ask "can I pick a better algorithm?", the honest answer is: yes, but the **bigger levers are the representation (features), the label definition, and the evaluation.** That is genuinely good news for us, because it is exactly where we can add value cheaply (see Sections 5-6).

> **One-line answer to your question:** No, VEP is not "just the algorithm in the back." It is a pipeline of five design choices, and the model is usually the part that matters *least*.


## 2. The fork in the road: *pathogenicity* vs *direction of effect* (this is our project's real identity)

This is the single most important thing to understand about our project, and it decides which "better methods" are even relevant.

**Most VEP tools answer one question:** *"Is this variant damaging / pathogenic?"* - a yes/no or a 0-1 score. That includes essentially all the famous ones: SIFT, PolyPhen-2, CADD, REVEL, EVE, ESM-1v, **AlphaMissense**.

**Our project answers a different, harder question:** *"Is this variant gain-of-function or loss-of-function?"* - the **direction** (or "mode of action") of the effect.

Why this matters:
- A **GOF** variant and a **LOF** variant can **both be pathogenic** - and a generic tool will often score them **the same** ("damaging"). It literally cannot tell them apart, because it was never trained to.
- The field now explicitly recognises this: *pathogenic missense variants in the same gene can act through different modes of action* [PreMode 2025; MissION 2025]. Predicting that mode is a separate, younger research area.

**Consequences for us:**
1. We **cannot** just plug in AlphaMissense / ESM and declare victory - they do not output GOF vs LOF.
2. But we **can** use their scores as **input features** (they capture "how impactful is this change?", which is still useful signal for direction).
3. The methods we should benchmark against are **not** the generic pathogenicity tools but the **direction-of-effect** tools (Section 4).
4. Because GOF/LOF prediction - especially for **ligand-gated** channels like nAChRs - is under-explored, there is real room for a genuine contribution (Section 5).


## 3. A map of VEP method families

Six broad families. For each: what it predicts, whether it needs variant labels, and how well it fits **our** small, ion-channel, GOF/LOF problem.

| Family | Examples | Predicts | Needs labelled variants? | Fit for us |
|---|---|---|---|---|
| **(a) Supervised on engineered features** | PolyPhen-2, CADD, REVEL; **our current model**; the **VEP-ENaC** template | pathogenicity (mostly) | Yes | This is what we do now - solid, interpretable, works with small data |
| **(b) Zero-shot evolutionary / unsupervised** | SIFT, EVmutation, DeepSequence, **EVE**, GEMME, Tranception/TranceptEVE | a fitness/likelihood score | No (learns from an MSA of the protein family) | Great as an **added feature** or baseline; does not give direction |
| **(c) Protein language models** | **ESM-1v, ESM-2, ESM-C**, ProtT5 | zero-shot score, or **embeddings** | No (pretrained) | **High value**: use the wt->mut log-likelihood as a feature, or feed embeddings as features |
| **(d) Structure-based** | FoldX, Rosetta ddG, **RaSP**, ThermoMPNN, ESM-IF1, ProteinMPNN | stability change (ddG), or structural fitness | No (needs a 3D structure) | A ddG feature is a good **LOF correlate** (destabilising -> often LOF) |
| **(e) Ensembles / meta-predictors** | REVEL, BayesDel, MetaRNN, **PATHOS** (uses ESM-C) | pathogenicity | Yes | Pattern to copy: combine several scores; still pathogenicity, not direction |
| **(f) Direction-of-effect / mode-of-action** | **funNCion, LoGoFunc, PreMode, MissION** | **GOF vs LOF** (and neutral) | Yes | **This is our family.** See Section 4 |

**Take-away:** families (b), (c), (d) give us **better features**; family (f) gives us **methods, datasets, and benchmarks for the actual task**. Family (e) is a recipe (combine signals) we can borrow.


## 4. The methods that do *exactly* our task (must-reads)

These predict the **direction** of effect for missense variants - several specifically for **ion channels**. Read them in this order; the last one is the closest match to our project.

### funNCion - GOF/LOF for voltage-gated Na+ and Ca2+ channels
*Brunger et al., Science Translational Medicine 2020.*
- Trained on **518 LOF + 309 GOF** variants whose direction was inferred from patient phenotypes.
- Features: **sequence + structure** based (very similar in spirit to ours).
- Classic ML classifier, **ROC-AUC ~= 0.85**; predictions matched molecular tests for 87 functionally studied variants.
- **Why read it:** it is the original proof that "GOF vs LOF from engineered features" works for ion channels - our exact recipe, on neighbouring channels.

### Voltage-gated K+ channel predictor (multi-task learning)
*EBioMedicine 2022.* Multi-task learning across potassium-channel variant effects - shows the value of sharing signal across related tasks/genes when per-gene data is scarce (a problem we have).

### LoGoFunc - genome-wide GOF / LOF / neutral
*Stein et al., Genome Medicine 2023.*
- **Ensemble** model over **474 features**, including **AlphaFold2** structural features and protein-protein-interaction **network** features.
- Three-way output (GOF / LOF / neutral); **precomputed for ~71 million** missense variants (free lookup at the GOF/LOF web app).
- **Why read it:** the most complete feature catalogue for direction prediction - a shopping list of features we could add - and it shows direction-specific tools beat generic pathogenicity tools at finding GOF/LOF.

### PreMode - mode-of-action by graph deep learning
*Nature Communications 2025 (bioRxiv 2024).*
- Predicts **mode of action** (incl. GOF/LOF) using a **graph neural network** over protein **sequence + structural context**, with **gene-specific** fine-tuning.
- **Why read it:** the modern deep-learning take on our task; shows how structure context is encoded as a graph. (Heavier to run; useful for ideas even if we keep a simpler model.)

### MissION - ion-channel missense GOF/LOF with a protein language model  *(closest to us; Oct 2025)*
*"Functional Effect Predictions For Ion Channel Missense Variants Using a Protein Language Model", medRxiv 2025.*
- Trained on **3,176 GOF/LOF variants across 47 ion-channel genes** - the **largest** such set to date.
- Representation = **ESM protein-language-model embeddings** (they tried ESM sizes from 8M up to 3B parameters) **plus** structural + sequence features, **phenotype annotations mapped to HPO terms**, and **GO terms**.
- Model = a **neural network** trained with **focal loss** to cope with class imbalance.
- **ROC-AUC = 0.925**, beating the previous best (0.897).
- **Why this is the most important paper for you:**
  1. It is *exactly* our task (ion-channel missense, GOF vs LOF).
  2. It validates the ingredients we already suspected we need: **pLM embeddings + structure + sequence + imbalance handling.**
  3. **Check whether its 47 genes include the nAChR genes (CHRNA*/CHRNB*/CHRND/E/G).** If nAChRs are absent or barely covered, then a **nAChR-focused** predictor (with our hand-curated data and the open/closed structural feature from `week1.ipynb`) is a real, defensible niche rather than a re-run of MissION.

> **Big-picture take-away:** our approach is mainstream and validated; the frontier for *our task* has moved to **(pLM embeddings + structure + phenotype) with class-imbalance handling**, evaluated honestly. None of that requires abandoning what we have - it is mostly *adding better features* to the pipeline we already built.


## 5. Can we build something newer / better? Concrete options, ranked for *our* setup

**Reality check first.** We have **~351 variants**. That rules out training large deep models from scratch (PreMode/MissION lean on thousands of variants and/or huge pretrained models). The winning strategy at our size is **transfer learning**: let someone else's giant pretrained model do the heavy lifting, and feed its outputs as **features** into our existing, well-behaved classifier. Everything below respects that.

| # | What to add / try | Why it should help | Effort | Novelty |
|---|---|---|---|---|
| 1 | **Zero-shot scores as features**: ESM-1v wt->mut log-likelihood, AlphaMissense, EVE | Strong general "impact" signal; cheap; standard practice | Low | Low |
| 2 | **ESM-2 embeddings** of the mutation window as features (as MissION does) | Captures sequence/structure context the hand features miss | Medium | Medium |
| 3 | **Structure-based ddG** feature (RaSP or ThermoMPNN) | Destabilisation correlates with LOF -> direction signal | Medium | Medium |
| 4 | **Class-imbalance handling**: focal loss / balanced weights / resampling | We are 218 LOF vs 133 GOF; MissION credits focal loss for its gains | Low | Low |
| 5 | **Borrow funNCion / LoGoFunc feature sets** (network features, finer structural features) | Proven informative for direction prediction | Medium | Low |
| 6 | **The novel angle**: a nAChR-specific GOF/LOF model = pLM score + the **open/closed conformational feature** from `week1.ipynb` + our curated data | Combines a modern representation with a mechanism-aware, nAChR-specific structural feature that generic tools lack | Medium-High | **High (paper-worthy)** |

**Recommended path:** do #1 and #4 immediately (low effort, likely measurable gain), then #2 and #3, and frame the whole thing around #6 as the contribution. Prove each addition with an **ablation** (see `week1.ipynb`, Section 5) so we know which features actually earned their place.

**A note on "algorithms" specifically.** Yes, we can also try different classifiers (we already have logistic regression, SVM, random forest, LightGBM, XGBoost in the repo). But per Section 1, expect the algorithm swap to move F1 by less than a good new feature does. Spend effort on representation first.


## 6. What "more precise" means - and how to prove it honestly

"Better/more precise" is only meaningful against a fair test. Three things to get right:

**1. Compare against the right baselines.**
- A **trivial baseline** (always predict the majority class = LOF) - any real model must beat it clearly.
- A **zero-shot baseline** (ESM-1v score thresholded) - if our 52-feature model cannot beat a single free pretrained score, that is important to know.
- The **direction-of-effect** tools (funNCion/LoGoFunc/MissION) on a comparable set - not the generic pathogenicity tools.

**2. Use the right benchmark, and know its limits.**
- **ProteinGym** is the field-standard benchmark (>2.5M variant measurements, 217 deep-mutational-scanning assays; leaderboard of zero-shot and supervised methods). **But** it scores *fitness/pathogenicity*, **not GOF vs LOF** - so it is the benchmark for families (b)/(c), not directly for our task. For us, the meaningful comparison is GOF/LOF performance reported by funNCion/LoGoFunc/MissION.

**3. Avoid data leakage - the number-one way VEP papers fool themselves.**
- Split by **position / protein / structure group**, not random rows, so the model cannot "memorise" a residue it also saw in training (ties to the subunit-grouped CV point in `week1.ipynb`).
- Watch the subtler trap: features from **pretrained models** (AlphaMissense, ESM) may have been trained on variants that end up in your test set - so a high score can be partly circular. Recent work explicitly targets this (e.g. **StructGuy 2025**, "data-leakage-free prediction of functional effects").

**4. Report metrics suited to imbalance.** ROC-AUC, **PR-AUC**, balanced accuracy, and **MCC** - not plain accuracy (which a majority-class guesser can inflate).


## 7. The papers you found - how to share them so I can review them

Yes, please share them - reading the specific papers you found is the fastest way to judge whether there is a genuinely new method we can adopt or improve on. Here is the most reliable way to get them to me:

**Best option - drop the PDFs in a folder.**
- Put the PDFs anywhere under the project, e.g. a new `papers/` folder inside `VEP Nachr`, and tell me the path (or just the filenames).
- I can open PDFs directly and read them page by page, then summarise each and assess fit to our nAChR GOF/LOF problem.

**Also fine - paste the links or DOIs.**
- I can fetch most journal/arXiv/bioRxiv pages directly.
- Caveat: some preprint servers (e.g. **medRxiv**) block automated fetching (that happened while researching this notebook), so for those the **PDF-in-a-folder** route is more reliable.

**What I will do with each paper:**
1. One-paragraph plain summary (what it predicts, data, representation, model, results).
2. Where it sits in the map in Section 3.
3. Concretely, **what we could borrow** (features, label scheme, evaluation) and **what we could do better / differently** for nAChRs.
4. A verdict: ignore / use as a feature source / benchmark against / try to beat.

**Papers under review** *(to fill in once you share them):*

| # | Title / DOI | Family (Sec. 3) | Relevant to us? | What to borrow | Verdict |
|---|---|---|---|---|---|
| 1 | *(paste here)* | | | | |
| 2 | | | | | |
| 3 | | | | | |


## 8. Summary and recommended next steps

**Your three questions, answered:**
1. **Is VEP just the algorithm?** No. It is a five-part pipeline (target, data, representation, model, evaluation) and the algorithm is usually the *least* decisive part. The big wins come from better **representations** - which is exactly where we can add value cheaply.
2. **Can we try newer/better methods?** Yes, and we should - but as **added features via transfer learning** (ESM scores/embeddings, AlphaMissense, structural ddG), not by training a giant model on 351 variants. There is also a small family of methods doing *exactly* our task (Section 4) to benchmark against and learn from.
3. **The papers you found:** share them as PDFs in a `papers/` folder (most reliable) or as links/DOIs, and I will review each against our problem (Section 7).

**Recommended next steps (in order):**
1. Read **MissION** (Oct 2025) and **funNCion** - the two closest to our task - and check whether MissION's 47 genes include nAChRs. *(This decides how novel a nAChR-specific model is.)*
2. Add an **ESM-1v zero-shot score** and an **AlphaMissense score** as features; re-run with an **ablation** to measure the gain.
3. Add **class-imbalance handling** (focal loss / balanced weights).
4. Frame the project's contribution as a **nAChR-specific GOF/LOF predictor** that combines a protein-language-model representation with the **open/closed conformational structural feature** from `week1.ipynb`.
5. Lock in **honest evaluation**: grouped CV, imbalance-aware metrics, baselines, leakage checks.

**Open questions to discuss with your advisor:**
1. Are we allowed to use **pretrained large models** (ESM, AlphaMissense) as feature sources, or must the model stay fully hand-engineered/interpretable?
2. Is the goal to **match** the VEP-ENaC methodology for comparability, or are we free to **diverge** toward the MissION-style (pLM + structure) approach?
3. Do we want a **two-way** (LOF/GOF) or **three-way** (LOF/GOF/neutral, like LoGoFunc) target? Our data has some "no net effect" rows we currently drop.


## References and links

*Citations anchor each claim; verify exact author lists / years against the originals before any formal write-up. Conformational and method facts here were checked against the sources below (June 2026).*

**Benchmarks and the VEP method landscape**
- ProteinGym - Notin P, et al. (2023). ProteinGym: large-scale benchmarks for protein fitness prediction and design. *NeurIPS*. https://proteingym.org
- Tranception / TranceptEVE - Notin P, et al. (2022). Tranception: protein fitness prediction with autoregressive transformers and inference-time retrieval. *ICML*. arXiv:2205.13760
- GEMME - Laine E, Karami Y, Carbone A (2019). *Mol Biol Evol* 36:2604-2619.
- VespaG / VESPA - Marquet C, et al. (2022-2024). Embedding-based variant effect prediction. https://github.com/Rostlab/VespaG
- StructGuy (data-leakage-free prediction) - bioRxiv 2025. https://www.biorxiv.org/content/10.64898/2025.12.01.691563

**Generic pathogenicity / fitness predictors (useful as feature sources)**
- AlphaMissense - Cheng J, et al. (2023). *Science* 381:eadg7492. https://www.science.org/doi/10.1126/science.adg7492
- ESM-1v - Meier J, et al. (2021). Language models enable zero-shot prediction of mutation effects. *NeurIPS* / bioRxiv 2021.07.09.450648
- ESM-2 / ESMFold - Lin Z, et al. (2023). *Science* 379:1123-1130. https://github.com/facebookresearch/esm
- EVE - Frazer J, et al. (2021). *Nature* 599:91-95.
- PATHOS (uses ESM Cambrian / ESM-C) - medRxiv 2025. https://www.medrxiv.org/content/10.64898/2025.12.22.25342839

**Structure-based stability (LOF correlate)**
- RaSP - Blaabjerg LM, et al. (2023). Rapid protein stability prediction using deep learning representations. *eLife* 12:e82593.
- ThermoMPNN - Dieckhaus H, et al. (2024). *PNAS* 121:e2314853121.
- ProteinMPNN - Dauparas J, et al. (2022). *Science* 378:49-56.

**Direction-of-effect / mode-of-action (our task)**
- funNCion - Brunger T, et al. (2020). Predicting functional effects of missense variants in voltage-gated sodium and calcium channels. *Science Translational Medicine* 12:eaay6848. https://www.science.org/doi/10.1126/scitranslmed.aay6848
- Voltage-gated K+ channel multi-task predictor - *EBioMedicine* (2022). https://www.sciencedirect.com/science/article/pii/S2352396422002961
- LoGoFunc - Stein D, et al. (2023). Genome-wide prediction of pathogenic gain- and loss-of-function variants from ensemble learning of a diverse feature set. *Genome Medicine* 15:103. https://link.springer.com/article/10.1186/s13073-023-01261-9 ; web app: https://itanlab.shinyapps.io/goflof/
- PreMode - (2025). PreMode predicts mode-of-action of missense variants by deep graph representation learning. *Nature Communications*. https://www.nature.com/articles/s41467-025-62318-4
- MissION - (2025). Functional Effect Predictions For Ion Channel Missense Variants Using a Protein Language Model. *medRxiv* 2025.10.16.25337735. https://www.medrxiv.org/content/10.1101/2025.10.16.25337735v1
